In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import spearmanr
import nbformat

from clean_dataset import nettoyer
from ratio_fixe_graph import courbe_avec_outliers


original = pd.read_csv('data/life_expectancy_data.csv')
df = nettoyer(original)

Colonnes supprimées : ['BMI', 'HIV/AIDS', 'Population', 'percentage expenditure']
NaN ajoutés par le nettoyage :
Adult Mortality                    122
Alcohol                            181
Hepatitis B                         51
Polio                               34
Total expenditure                   26
Diphtheria                          39
GDP                                208
thinness 1-19 years                 52
thinness 5-9 years                  52
Income composition of resources    130
Schooling                           28


### **Les outils d'aide à la création de graphiques**

Ici nous avons créé un fichier de détection des zéros. En effet nous devons être en mesure de décider si les zéros doivent être transformés en NaN (par exemple pour ne pas influencer le calcul de la médiane, éléments central du calcul du ratio plus loin), ou rester à zéro (comme élement relevé dans un pays : par exemple zéro cas de rougeole).

In [4]:
import numpy as np
import pandas as pd


def diagnostic_zeros(df, colonne, mini_valeurs=3):
    """Diagnostic des zéros et des séries figées pour UNE colonne.

    Affiche un bilan global, puis le détail par pays.
    mini_valeurs : nombre minimum de valeurs renseignées pour tester
                   si une série est figée (constante).
    """
    serie = df[colonne]

    # --- Bilan global ---
    nb_zeros = (serie == 0).sum()
    print(f"--- {colonne} ---")
    print(f"Zéros : {nb_zeros} ({100 * nb_zeros / len(serie):.1f} %)")
    print(f"NaN   : {serie.isna().sum()}")

    if nb_zeros == 0:
        print("Aucun zéro à convertir.\n")
        return None

    # --- Détail par pays ---
    lignes = []
    for pays, bloc in df.groupby("Country"):
        bloc = bloc.sort_values("Year")        # ordre chronologique indispensable
        valeurs = bloc[colonne]
        est_zero = (valeurs == 0).values       # tableau de True/False

        if not est_zero.any():
            continue                           # ce pays n'a aucun zéro, on passe

        # nb_renseignees : valeurs non NaN (contrairement à len(), qui compte les NaN)
        nb_renseignees = valeurs.notna().sum()

        # Les zéros sont-ils tous collés au début de la série ?
        # est_zero[:n].all() = les n premières valeurs sont toutes des zéros
        nb_zeros_pays = est_zero.sum()
        debut_serie = est_zero[:nb_zeros_pays].all()

        # Série figée : toutes les valeurs renseignées sont identiques
        # (on ignore les zéros pour ce test, sinon ils fausseraient le compte)
        sans_zero = valeurs.replace(0, np.nan)
        figee = (sans_zero.notna().sum() >= mini_valeurs
                 and sans_zero.nunique() == 1)

        lignes.append({
            "pays": pays,
            "nb_zeros": nb_zeros_pays,
            "nb_renseignees": nb_renseignees,
            "tout_a_zero": nb_zeros_pays == nb_renseignees,
            "debut_serie": debut_serie,
            "figee_hors_zeros": figee,
        })

    detail = pd.DataFrame(lignes).set_index("pays")

    # --- Synthèse : c'est elle qui oriente la décision ---
    print(f"Pays touchés        : {len(detail)} / {df['Country'].nunique()}")
    print(f"  dont tout à zéro  : {detail['tout_a_zero'].sum()}")
    print(f"  dont zéros en début de série : {detail['debut_serie'].sum()}")
    print(f"  dont série figée hors zéros  : {detail['figee_hors_zeros'].sum()}")
    print()
    print(detail.to_string())
    print()
    return detail

diagnostic_zeros(df, "Hepatitis B")

--- Hepatitis B ---
Zéros : 0 (0.0 %)
NaN   : 604
Aucun zéro à convertir.



Voici un code qui nous permet de détecter où se trouve le ratio idéal à appliquer à la colonne, pour la création d'histogramme par pays.    

In [5]:
def ratio_mediane(df, colonne):
    """Ratio valeur / médiane du pays, pour chaque ligne."""
    # Médiane de chaque pays (les NaN sont ignorés par la médiane)
    mediane = df.groupby("Country")[colonne].transform("median")
    # Si la médiane vaut 0, la division donne inf : on remplace par NaN
    return (df[colonne] / mediane).replace([np.inf, -np.inf], np.nan)


def _exemples(df, colonne, ratio, masque, n):
    """Choisit n lignes réparties dans la tranche et les décrit avec leur contexte."""
    # On trie par pays/année pour que "précédente" et "suivante" aient un sens
    d = df.sort_values(["Country", "Year"]).copy()
    d["ratio"] = ratio.loc[d.index]
    d["mediane"] = d.groupby("Country")[colonne].transform("median")
    d["avant"] = d.groupby("Country")[colonne].shift(1)    # valeur de la ligne d'avant
    d["apres"] = d.groupby("Country")[colonne].shift(-1)   # valeur de la ligne d'après
 
    # Lignes de la tranche, triées par ratio croissant
    t = d[masque.loc[d.index]].sort_values("ratio")
 
    # n lignes régulièrement espacées (la plus basse, une du milieu, la plus haute...)
    positions = np.unique(np.linspace(0, len(t) - 1, min(n, len(t))).astype(int))
    texte = []
    for _, l in t.iloc[positions].iterrows():
        texte.append(f"{l['Country']} {int(l['Year'])} : {l['avant']:g} -> [{l[colonne]:g}] -> "
                     f"{l['apres']:g}  (médiane {l['mediane']:g})")
    return texte
 
 
def tableau_ratio(df, colonne, bornes=None, exemples=3):
    """Nombre de lignes par tranche de ratio, avec quelques exemples par tranche.
 
    Un exemple se lit : Pays Année : valeur d'avant -> [valeur de la ligne] -> valeur d'après (médiane du pays).
    Une valeur isolée entre deux valeurs proches = probable erreur ; une série qui monte
    ou descend d'un coup = probable vraie évolution.
    """
    ratio = ratio_mediane(df, colonne)
 
    if bornes is None:
        bornes = ([-0.001] + [round(0.05 * i, 2) for i in range(1, 21)]
                  + [1.25, 1.5, 2, 3, 5, float("inf")])
 
    tranche = pd.cut(ratio, bornes)
    comptes = tranche.value_counts().sort_index()
 
    for intervalle, n in comptes.items():
        print(f"{str(intervalle):<16}{n:>5} lignes")
        if n > 0 and exemples > 0:
            for ligne in _exemples(df, colonne, ratio, tranche == intervalle, exemples):
                print("      ", ligne)
    print("Ratio non calculable (NaN ou médiane = 0) :", ratio.isna().sum())
    
def histogramme_ratio(df, colonne, max_ratio=3, pas=0.05):
    """Histogramme du ratio valeur / médiane du pays.
    Barres de largeur `pas` ; les ratios > max_ratio sont regroupés dans la dernière barre."""
    ratio = ratio_mediane(df, colonne).dropna()
 
    # Plafonner à max_ratio : la dernière barre regroupe tous les ratios >= max_ratio
    ratio = ratio.clip(upper=max_ratio)
 
    fig = go.Figure(go.Histogram(
        x=ratio,
        xbins=dict(start=0, end=max_ratio + pas, size=pas),   # barres de largeur `pas`
    ))
    fig.update_layout(title=f"{colonne} : valeur / médiane du pays",
                      xaxis_title=f"Ratio (plafonné à {max_ratio})",
                      yaxis_title="Nombre de lignes")
    return fig


### **1) Traitement de la colonne Adult Mortality**

In [6]:

diagnostic_zeros(df, "Adult Mortality")
tableau_ratio(df, "Adult Mortality")
histogramme_ratio(df, 'Adult Mortality')
histogramme_ratio(df, 'Adult Mortality')


--- Adult Mortality ---
Zéros : 0 (0.0 %)
NaN   : 132
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         1 lignes
       Albania 2007 : 99 -> [9] -> 50  (médiane 62)
(0.15, 0.2]         1 lignes
       Albania 2000 : nan -> [11] -> 14  (médiane 62)
(0.2, 0.25]         3 lignes
       Albania 2001 : 11 -> [14] -> 15  (médiane 62)
       Albania 2002 : 14 -> [15] -> 18  (médiane 62)
       Albania 2005 : 17 -> [15] -> 99  (médiane 62)
(0.25, 0.3]         2 lignes
       Albania 2004 : 18 -> [17] -> 15  (médiane 62)
       Albania 2003 : 15 -> [18] -> 17  (médiane 62)
(0.3, 0.35]         0 lignes
(0.35, 0.4]         0 lignes
(0.4, 0.45]         0 lignes
(0.45, 0.5]         0 lignes
(0.5, 0.55]         1 lignes
       Zimbabwe 2015 : 371 -> [336] -> nan  (médiane 646.167)
(0.55, 0.6]         2 lignes
       Zimbabwe 2014 : 399 -> [371] -> 336  (médiane 646.167)
       Botswana 2015 : 268 -> [256] -> nan  (médiane 431.5)
(0.6, 0.65]       

Ici nous  avons un creux à 0.5, nous utiliserons donc un ratio de 0.5

In [7]:
courbe_avec_outliers(df, 'Afghanistan', 'Adult Mortality', "Mort adulte 15 à 60 ans", 0.5)

### **2) Traitement de la colonne Alcohol**

In [8]:
diagnostic_zeros(df, "Alcohol")
tableau_ratio(df, "Alcohol")

--- Alcohol ---
Zéros : 0 (0.0 %)
NaN   : 375
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         2 lignes
       Timor-Leste 2006 : 0.36 -> [0.04] -> 0.05  (médiane 0.37)
       Timor-Leste 2007 : 0.04 -> [0.05] -> 0.06  (médiane 0.37)
(0.15, 0.2]         1 lignes
       Timor-Leste 2008 : 0.05 -> [0.06] -> 0.09  (médiane 0.37)
(0.2, 0.25]         1 lignes
       Timor-Leste 2009 : 0.06 -> [0.09] -> 0.1  (médiane 0.37)
(0.25, 0.3]         3 lignes
       Brunei Darussalam 2004 : 0.12 -> [0.11] -> 0.16  (médiane 0.42)
       Timor-Leste 2010 : 0.09 -> [0.1] -> 0.5  (médiane 0.37)
       Brunei Darussalam 2003 : 0.13 -> [0.12] -> 0.11  (médiane 0.42)
(0.3, 0.35]         6 lignes
       Brunei Darussalam 2002 : 0.47 -> [0.13] -> 0.12  (médiane 0.42)
       Pakistan 2002 : 0.01 -> [0.01] -> 0.01  (médiane 0.03)
       Pakistan 2014 : 0.04 -> [0.01] -> nan  (médiane 0.03)
(0.35, 0.4]         5 lignes
       Zimbabwe 2000 : nan -> [1.68] ->

Nous constatons un creux vers 0.3, que nous utiliserons comme ratio.

### **3) Traitement de la colonne Hepatitis B**

In [9]:
diagnostic_zeros(df, "Hepatitis B")
tableau_ratio(df, "Hepatitis B")

--- Hepatitis B ---
Zéros : 0 (0.0 %)
NaN   : 604
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         2 lignes
       India 2013 : 73 -> [7] -> 79  (médiane 37)
       Benin 2002 : nan -> [15] -> 73  (médiane 75.5)
(0.2, 0.25]         5 lignes
       Yemen 2000 : nan -> [14] -> 19  (médiane 69)
       India 2005 : nan -> [8] -> 15  (médiane 37)
       Equatorial Guinea 2014 : nan -> [2] -> 16  (médiane 9)
(0.25, 0.3]         5 lignes
       Yemen 2001 : 14 -> [19] -> 31  (médiane 69)
       Togo 2008 : nan -> [24] -> 78  (médiane 84)
       Djibouti 2007 : nan -> [25] -> 88  (médiane 84)
(0.3, 0.35]         5 lignes
       Saint Vincent and the Grenadines 2003 : nan -> [31] -> 99  (médiane 99)
       Sri Lanka 2003 : nan -> [32] -> 62  (médiane 99)
       Comoros 2003 : nan -> [28] -> 77  (médiane 81)
(0.35, 0.4]         8 lignes
       Ukraine 2011 : 48 -> [21] -> 46  (médiane 57)
       Ukraine 2014 : 46 

Nous utiliserons ici un ratio de 0,2.

In [10]:
courbe_avec_outliers(df, 'France', 'Hepatitis B', "Taux de couverture vaccinale contre l'Hépatite B chez les enfants de 1 an (exprimé en %)", 0.3)

### **4) Traitement de la colonne Measles**

In [26]:
diagnostic_zeros(df, "Measles")
tableau_ratio(df, "Measles")
histogramme_ratio(df, 'Measles')


--- Measles ---
Zéros : 983 (33.5 %)
NaN   : 0
Pays touchés        : 143 / 193
  dont tout à zéro  : 30
  dont zéros en début de série : 37
  dont série figée hors zéros  : 1

                                       nb_zeros  nb_renseignees  tout_a_zero  debut_serie  figee_hors_zeros
pays                                                                                                       
Albania                                       5              16        False        False             False
Algeria                                       3              16        False        False             False
Antigua and Barbuda                          16              16         True         True             False
Argentina                                    10              16        False        False             False
Armenia                                       4              16        False        False             False
Australia                                     1              16     

Les données de la colonne Measles sont censé être un nombre de cas de rougeole pour 1000 personnes. Or beaucoup de mesures dépassent 1000. Après une vérification sur Internet, il est confirmé que ces chiffres sont un nombre de cas. Par ailleurs, la distribution des données ne permet pas de détecter un creux, il n'y aura pas d'utilisation d'un ratio pour un graphique : nous n'allons donc appliquer aucun changement sur ces données, nous les gardons telle qu'elle.

### **5) Traitement de la colonne under_five deaths**

In [12]:
diagnostic_zeros(df, "under-five deaths")
tableau_ratio(df, "under-five deaths")
histogramme_ratio(df, 'under-five deaths')

--- under-five deaths ---
Zéros : 785 (26.7 %)
NaN   : 0
Pays touchés        : 64 / 193
  dont tout à zéro  : 55
  dont zéros en début de série : 55
  dont série figée hors zéros  : 8

                                           nb_zeros  nb_renseignees  tout_a_zero  debut_serie  figee_hors_zeros
pays                                                                                                           
Albania                                           1              16        False        False              True
Antigua and Barbuda                              16              16         True         True             False
Austria                                          16              16         True         True             False
Bahamas                                          16              16         True         True             False
Bahrain                                          16              16         True         True             False
Barbados                       

Nous  choisissons de ne pas modifier les données d'"under-five deaths" car les valeurs sont trop petites pour rendre un ratio pertinent. En effet les zéros sont en fait des valeurs de petits pays, l'unité est mal choisie pour mesurer cette donnée.     
Nous avons supprimé la colonne "infant deaths" car nous sommes partis du principe que ces valeurs étaient comprises dans la colonne "under-five deaths" : aucune valeurs de "infant deaths" ne dépasse "under-five deaths". Afin d'aller dans le sens de cette hypothèse, nous effectuons un test de corrélation entre ce deux colonnes. Par ailleurs ce code nous permet de calculer la part d'"infant deaths" dans "under-five deaths". La max de "1" montre que, sur une ligne, le maximum d'"infant deaths" est exactement "under-five deaths" : 

In [13]:
def comparer_colonnes(df, a, b):
    """Compare deux colonnes : corrélation, et test « a est compris dans b » (a <= b)."""
    # On garde les lignes où les deux colonnes sont renseignées
    d = df[[a, b]].dropna()
    print(f"Lignes comparées : {len(d)}")

    # Corrélation : proche de 1 = les deux colonnes montent et descendent ensemble
    # Pearson : relation linéaire ; Spearman : ne regarde que l'ordre des valeurs
    print("Corrélation de Pearson  :", round(d[a].corr(d[b]), 4))
    print("Corrélation de Spearman :", round(d[a].corr(d[b], method="spearman"), 4))

    # Test d'inclusion : si a est compris dans b, a ne peut jamais dépasser b
    print(f"Lignes où {a} > {b} :", (d[a] > d[b]).sum())

    # Part de a dans b (seulement là où b est positif, sinon division par 0)
    positif = d[d[b] > 0]
    print(f"Part de {a} dans {b} :")
    print((positif[a] / positif[b]).describe().round(3).to_string())

comparer_colonnes(df,"infant deaths", "under-five deaths")

Lignes comparées : 2938
Corrélation de Pearson  : 0.9966
Corrélation de Spearman : 0.9932
Lignes où infant deaths > under-five deaths : 0
Part de infant deaths dans under-five deaths :
count    2153.000
mean        0.774
std         0.198
min         0.000
25%         0.667
50%         0.778
75%         0.909
max         1.000


### **6) Traitement de la colonne Polio**

In [14]:
diagnostic_zeros(df, "Polio")
tableau_ratio(df, "Polio")
histogramme_ratio(df, 'Polio', max_ratio=3, pas=0.05)

--- Polio ---
Zéros : 0 (0.0 %)
NaN   : 53
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         1 lignes
       Equatorial Guinea 2015 : 24 -> [17] -> nan  (médiane 45)
(0.4, 0.45]         2 lignes
       Afghanistan 2000 : nan -> [24] -> 35  (médiane 58)
       Samoa 2004 : 82 -> [26] -> 55  (médiane 58.75)
(0.45, 0.5]         5 lignes
       Congo 2000 : nan -> [31] -> 32  (médiane 68)
       Ukraine 2014 : 72 -> [45] -> 51  (médiane 93)
       Central African Republic 2013 : 47 -> [23] -> 47  (médiane 46)
(0.5, 0.55]         5 lignes
       Sierra Leone 2001 : 46 -> [38] -> 54  (médiane 72)
       Timor-Leste 2002 : nan -> [38] -> 55  (médiane 71.25)
       Ukraine 2015 : 45 -> [51] -> nan  (médiane 93)
(0.55, 0.6]        10 lignes
       Angola 2006 : 39 -> [36] -> 75  (médiane 64)
       S

Nous utiliserons un ratio de 2,5 car il se toruve au milieu du creux. Les données avec un ratio proche de zéros représentent 171 lignes.      

### **7) Traitement de la colonne Total expenditure**

In [25]:
diagnostic_zeros(df, "Total expenditure")
tableau_ratio(df, "Total expenditure")
histogramme_ratio(df, "Total expenditure", max_ratio=3, pas=0.05)

--- Total expenditure ---
Zéros : 0 (0.0 %)
NaN   : 252
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         1 lignes
       Liberia 2003 : 5.43 -> [3.44] -> 8.77  (médiane 9.39667)
(0.4, 0.45]         1 lignes
       Democratic Republic of the Congo 2000 : nan -> [1.45] -> 2.67  (médiane 3.53)
(0.45, 0.5]         1 lignes
       Lao People's Democratic Republic 2014 : 2 -> [1.87] -> nan  (médiane 3.77)
(0.5, 0.55]         4 lignes
       Timor-Leste 2006 : 1.5 -> [0.65] -> 0.695  (médiane 1.29)
       Malawi 2002 : 5.7 -> [4.82] -> 6.35  (médiane 9.31)
       Timor-Leste 2007 : 0.65 -> [0.695] -> 0.74  (médiane 1.29)
(0.55, 0.6]        11 lignes
       Rwanda 2002 : 4.38 -> [4.18] -> 6.34  (médiane 7.53)
       Timor-Leste 2008 : 0.695 -> [0.74] -> 1.2  (médiane 1.29)
       Kiribati 2013 : 1.

### **8) Traitement de la colonne Dyphteria**

In [16]:
diagnostic_zeros(df, "Diphtheria")
tableau_ratio(df, "Diphtheria")
histogramme_ratio(df, "Diphtheria", max_ratio=3, pas=0.05)

--- Diphtheria ---
Zéros : 0 (0.0 %)
NaN   : 58
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         1 lignes
       Afghanistan 2000 : nan -> [24] -> 33  (médiane 62.5)
(0.4, 0.45]         4 lignes
       Samoa 2008 : 53 -> [26] -> 49  (médiane 63.5)
       Equatorial Guinea 2015 : 18.6667 -> [16] -> nan  (médiane 37.5)
       Angola 2000 : nan -> [28] -> 38  (médiane 64)
(0.45, 0.5]         3 lignes
       Congo 2000 : nan -> [33] -> 31  (médiane 72.5)
       Central African Republic 2013 : 47 -> [23] -> 47  (médiane 47)
       Equatorial Guinea 2014 : 21.3333 -> [18.6667] -> 16  (médiane 37.5)
(0.5, 0.55]         9 lignes
       Sierra Leone 2001 : 44 -> [38] -> 53  (médiane 75)
       Liberia 2004 : 35 -> [31] -> 42.3333  (médiane 58.8333)
       Syrian Arab Republic 2014 : 41 -> [43] -> 41

### **9) Traitement de la colonne GDP**

In [17]:
diagnostic_zeros(df, "GDP")
tableau_ratio(df, "GDP")
histogramme_ratio(df, "GDP", max_ratio=3, pas=0.05)

--- GDP ---
Zéros : 0 (0.0 %)
NaN   : 656
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         6 lignes
       Trinidad and Tobago 2009 : 21188.1 -> [1458.88] -> 16683.9  (médiane 9629.42)
       Oman 2006 : 12377 -> [1448.55] -> 1585.13  (médiane 8629.12)
       Oman 2007 : 1448.55 -> [1585.13] -> 2275.35  (médiane 8629.12)
(0.2, 0.25]        18 lignes
       Romania 2000 : nan -> [1668.16] -> 1839.73  (médiane 8228.07)
       Romania 2001 : 1668.16 -> [1839.73] -> 2124.87  (médiane 8228.07)
       Equatorial Guinea 2009 : 22742.4 -> [1653.29] -> 17136.4  (médiane 6628.88)
(0.25, 0.3]        17 lignes
       Myanmar 2005 : nan -> [247.243] -> 296.973  (médiane 987.737)
       Nigeria 2002 : nan -> [459.457] -> 512.657  (médiane 1619.75)
       Norway 2015 : 8824.74 -> [7455.25] -> nan  (médiane 24855.2)
(0.3, 0.35]        22 lignes
       Tajikistan 2000 : nan -> [138.437] -> 163.912  (médiane 460.729)
    

In [18]:
courbe_avec_outliers(df, 'Belarus', 'GDP', 'GDP Lituanie', 0.3)


### **10) Traitement de la Thiness 1-19 years**

Cette colonne reporte en fait le pourcentage d'enfants agés de 10 à 19 ans en état de maigreur.

In [19]:
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)
diagnostic_zeros(df, "thinness 1-19 years")
tableau_ratio(df, "thinness 1-19 years")
histogramme_ratio(df, "thinness 1-19 years", max_ratio=3, pas=0.05)

--- thinness 1-19 years ---
Zéros : 0 (0.0 %)
NaN   : 86
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         0 lignes
(0.4, 0.45]         0 lignes
(0.45, 0.5]         8 lignes
       South Africa 2015 : 4.9 -> [4.4] -> nan  (médiane 9.3)
       Kiribati 2010 : 0.1 -> [0.1] -> 0.1  (médiane 0.2)
       Kiribati 2015 : 0.1 -> [0.1] -> nan  (médiane 0.2)
(0.5, 0.55]         1 lignes
       South Africa 2014 : 5.4 -> [4.9] -> 4.4  (médiane 9.3)
(0.55, 0.6]         1 lignes
       South Africa 2013 : 6 -> [5.4] -> 4.9  (médiane 9.3)
(0.6, 0.65]         2 lignes
       Swaziland 2015 : 4.3 -> [4] -> nan  (médiane 6.3)
       South Africa 2012 : 6.6 -> [6] -> 5.4  (médiane 9.3)
(0.65, 0.7]         4 lignes
       Lesotho 2015 : 5.8 -> [5.5] -> nan  (médiane 8.2)
       Swaziland 2014 : 4.5 -> [4.3] -

### **11) Traitement de la Thiness 5-9 years**

In [20]:
diagnostic_zeros(df, "thinness 5-9 years")
tableau_ratio(df, "thinness 5-9 years")
histogramme_ratio(df, "thinness 5-9 years", max_ratio=3, pas=0.05)

--- thinness 5-9 years ---
Zéros : 0 (0.0 %)
NaN   : 86
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         0 lignes
(0.4, 0.45]         0 lignes
(0.45, 0.5]         6 lignes
       South Africa 2015 : 5.9 -> [5.3] -> nan  (médiane 11.225)
       Samoa 2013 : 0.1 -> [0.1] -> 0.1  (médiane 0.2)
       Samoa 2015 : 0.1 -> [0.1] -> nan  (médiane 0.2)
(0.5, 0.55]         1 lignes
       South Africa 2014 : 6.5 -> [5.9] -> 5.3  (médiane 11.225)
(0.55, 0.6]         1 lignes
       South Africa 2013 : 7.3 -> [6.5] -> 5.9  (médiane 11.225)
(0.6, 0.65]         1 lignes
       Swaziland 2015 : 4.3 -> [4.1] -> nan  (médiane 6.5)
(0.65, 0.7]         7 lignes
       South Africa 2012 : 8 -> [7.3] -> 6.5  (médiane 11.225)
       Zimbabwe 2015 : 5.7 -> [5.5] -> nan  (médiane 8)
       Lesotho 2014 : 5.9 -> [

### **11) Traitement de la colonne Income composition of resources**

In [21]:
diagnostic_zeros(df, "Income composition of resources")
tableau_ratio(df, "Income composition of resources")
histogramme_ratio(df, "Income composition of resources", max_ratio=3, pas=0.05)

--- Income composition of resources ---
Zéros : 0 (0.0 %)
NaN   : 297
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         0 lignes
(0.4, 0.45]         0 lignes
(0.45, 0.5]         0 lignes
(0.5, 0.55]         0 lignes
(0.55, 0.6]         0 lignes
(0.6, 0.65]         0 lignes
(0.65, 0.7]         0 lignes
(0.7, 0.75]         2 lignes
       Rwanda 2000 : nan -> [0.318] -> 0.332  (médiane 0.431)
       Ethiopia 2001 : nan -> [0.283] -> 0.298  (médiane 0.378)
(0.75, 0.8]         6 lignes
       Rwanda 2001 : 0.318 -> [0.332] -> 0.343  (médiane 0.431)
       Mozambique 2000 : nan -> [0.291] -> 0.298  (médiane 0.366)
       Cambodia 2000 : nan -> [0.401] -> 0.412  (médiane 0.503)
(0.8, 0.85]        20 lignes
       Afghanistan 2001 : 0.338 -> [0.34] -> 0.341  (médiane 0.424)
       Angola 2000 : nan

### **13) Traitement de la colonne schooling**

In [22]:
import plotly.express as px

# Une ligne par pays : on calcule la scolarité moyenne de chaque pays
pays = df.groupby("Country", as_index=False).agg(
    Schooling_moy=("Schooling", "mean")
)

# On retire les pays sans valeur, puis on trie pour que le graphe soit lisible
pays = pays.dropna().sort_values("Schooling_moy")

# Pays en abscisses, scolarité moyenne en ordonnées
fig = px.bar(pays, x="Country", y="Schooling_moy")
fig.show()

### **14) Traitement de la colonne life expectancy**

In [23]:
diagnostic_zeros(df, "Life expectancy")
tableau_ratio(df, "Life expectancy")
histogramme_ratio(df, "Life expectancy", max_ratio=3, pas=0.05)

--- Life expectancy ---
Zéros : 0 (0.0 %)
NaN   : 10
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         0 lignes
(0.4, 0.45]         0 lignes
(0.45, 0.5]         0 lignes
(0.5, 0.55]         0 lignes
(0.55, 0.6]         1 lignes
       Haiti 2010 : 62.5 -> [36.3] -> 62.3  (médiane 61.95)
(0.6, 0.65]         0 lignes
(0.65, 0.7]         0 lignes
(0.7, 0.75]         1 lignes
       Eritrea 2000 : nan -> [45.3] -> 58.1  (médiane 61.7)
(0.75, 0.8]         2 lignes
       Zambia 2000 : nan -> [43.8] -> 44.6  (médiane 56.55)
       Zambia 2001 : 43.8 -> [44.6] -> 45.5  (médiane 56.55)
(0.8, 0.85]        12 lignes
       Rwanda 2000 : nan -> [48.3] -> 48.6  (médiane 60.3)
       Botswana 2001 : 47.8 -> [46.7] -> 46  (médiane 57.2)
       Zambia 2004 : 46.4 -> [47.9] -> 49.3  (médiane 56.55)
(0.85, 0

### **15) Traitement de la colonne statut**

In [24]:
diagnostic_zeros(df, "Status")



--- Status ---
Zéros : 0 (0.0 %)
NaN   : 0
Aucun zéro à convertir.

